![banner](./images/banner.png "banner")

[Fuente de la imagen](https://unsplash.com/photos/Hcfwew744z4)
<a name='toc' />

# <font color=#ac6240>Cuantificaci?n de la incertidumbre en la predicci?n de fallo card?aco con aprendizaje autom?tico supervisado</font>

### <font color=#ac6240>Tabla de Contenidos</font>

<a href=#pre>Requisitos previos</a><br>
1. <a href=#project-description>Descripci?n del proyecto</a><br>
    1.1 <a href=#project-description-intro>Introducci?n</a><br>
    1.2 <a href=#project-description-goal>Objetivo</a><br>
    1.3 <a href=#project-description-data>Datos y modelo de aprendizaje autom?tico</a><br>
    1.4 <a href=#project-description-calibracion>Calibraci?n</a><br>
    1.5 <a href=#project-description-software>Software</a><br>
2. <a href=#prep>Preparaci?n de los datos</a><br>
    2.1 <a href=#prep-clean> Limpieza de los datos</a><br>
    2.2 <a href=#prep-split> Division de los datos en conjuntos de datos de entrenamiento, validaci?n y prueba</a><br>
    2.3 <a href=#prep-encoding> Codificaci?n y normalizaci?n de las 'features'</a><br>
3. <a href=#model-ml>Aplicaci?n del modelo de aprendizaje autom?tico: Bosque Aleatorio</a><br>
    3.1 <a href=#model-metric> Selecci?n de la m?trica y del umbral de decisi?n </a><br>
    3.2 <a href=#model-RF> Entrenamiento, prueba, y evaluaci?n del bosque aleatorio </a><br>
4. <a href=#app-uq>Cuantificaci?n de incertidumbre</a><br>
    4.1 <a href=#uq-cp> Conformal Predictors </a><br>
    4.2 <a href=#uq-ivap> Intervalos de predicci?n de fallo card?aco con Inductive Venn-ABERS Prediction </a><br>
    4.3 <a href=#uq-others> Otros m?todos de UQ para aprendizaje supervisado </a><br>
    4.4 <a href=#miscelanea> Miscel?nea </a><br>
    
5. <a href=#uq-end>Conclusi?n sobre la cuantificaci?n de incertidumbre</a><br>

Antes de nada, me gustar?a agradecer a les expertes [Maria Navarro](https://www.linkedin.com/in/maria-navarro-jimenez/) y a [Valeriy Manokhin](https://www.linkedin.com/in/valeriy-manokhin-phd-mba-cqf-704731236/) su inestimable ayuda en la elaboracion de este Jupyer Notebook. Pod?is encontrar las referencias a sus trabajos acreditadas a lo largo de este taller.

<font size="5">?? </font>

<a name='pre'/>

## <font color=#ac6240>Requisitos previos</font>

Este Jupyter Notebook puede recorrerse de manera independiente aunque es la continuaci?n de los Jupyter Notebooks de [este repositorio](https://github.com/MMdeCastro/Uncertainty_Quantification_XAI):
+ `Exploration_and_Classification.ipynb` donde explicamos c?mo explorar exhaustivamente los datos de pacientes que sufrieron o no fallo card?aco y c?mo evaluar la calidad de un modelo de aprendizaje autom?tico supervisado con m?tricas de rendimiento; elegimos el modelo que mejor se comportaba para este proyecto de predicci?n de fallo card?aco y era un bosque aleatorio, y
+ `XAI.ipynb` donde aplicamos varios m?todos de explicabilidad sobre las predicciones de ese bosque aleatorio para entender en qu? se fij? el modelo para clasificar a les pacientes.

En el README.md del repositorio del taller est?n los enlaces a varias charlas sobre UQ, en particular aconsejamos mirar ?stas dos presentaciones (contienen animaciones, mejor verlas en modo presentaci?n): 

+ [Intro to UQ with Conformal Prediction](https://docs.google.com/presentation/d/1Q6oxcgmNv0GsmNFA5npAzQGDuy_XolBKStN4KUhP4Gk/edit#slide=id.g2c6fde0c8c9_0_0) with a multiclass example (in English).
+ [Key Concepts of Conformal Prediction](https://docs.google.com/presentation/d/1bQYIFyQysQPx79wJq1mltylsH_aeO2AnLYRlHiLq6vo/edit#slide=id.g2dc32e8043a_0_5) (in English).


<font size="5"> ????</font>

<a name='project-description'/>

##  <font color=#ac6240>1. Descripci?n del proyecto</font>

<a name='project-description-intro'/>

### <font color=#ac6240>1.1 Introducci?n</font>

Estamos acostumbrades a modelos de aprendizaje autom?tico que nos devuelven una predicci?n en forma de un ?nico valor puntual ('single-value prediction'). Estos modelos no tienen ninguna manera de manifestar si est?n muy seguros o no de lo que predicen, por ejemplo, proporcionando una distribuci?n o rango de resultados razonables, para que podamos representar la predicci?n con unas 'barras de error' en lugar de con un s?lo punto. 

La falta de informaci?n sobre la certeza que tiene un modelo sobre los resultados que arroja es especialmente problem?tico cuando:
+ el modelo no es interpretable, es una 'black box' o caja opaca, (explicamos estos conceptos en el Jupyter Notebook anterior `XAI.ipynb`), o
+ pedimos al modelo que prediga un resultado sobre una instancia an?mala ('outlier') que no se parece a los datos hist?ricos con los que ese modelo fue entrenado, algo relacionado con el fen?meno denominado ['drift'](https://en.wikipedia.org/wiki/Drift_(data_science)) (m?s info sobre la definici?n de 'deriva' o desplazamiento 'shift' y sus diferentes tipos, [aqu?](https://analyticsindiamag.com/concept-drift-vs-data-drift-in-machine-learning/)), lo cual es indeseado y puede pasar f?cilmente desapercibido. 

Por ?ltimo remarcar que medir la certeza de las predicciones es especialmente ?til en aplicaciones sensibles o de riesgo donde desarrollar algoritmos robustos y confiables es esencial. Seg?n el acta reguladora de la inteligencia artificial de la Uni?n Europea, [The AI Act](https://artificialintelligenceact.eu/the-act/):

    "The purpose of this Regulation is to improve the functioning of the internal market and promoting the uptake of human centric and trustworthy artificial intelligence".

Como ejemplo sencillo, supongamos que entrenamos una red neuronal artificial (que no es interpretable) con im?genes de perres y gates con la tarea de distinguir entre perres y gates. Digamos que el umbral de decisi?n es tal que las instancias, es decir, las fotos, cuya 'raw score' sea mayor de 0.5 ser?n clasificadas como gates y el resto como perres, como en una t?pica clasificaci?n binaria. Supongamos adem?s que conseguimos que la red neuronal artificial tenga un rendimiento excelente sobre los datos de validaci?n y testado, puede que incluso tenga un 99.5% en 'accuracy' sobre ambos conjuntos. Ahora, aplicamos la red y le pedimos que clasifique estas fotos:

![cat_dog_bird](./images/cat_dog_bird.png "cat_dog_bird")

[Fuente de la imagen: _"Are you sure about that?! Uncertainty Quantification in AI"_](https://florianwilhelm.info/2019/10/uncertainty_quantification_in_ai/)

Y vemos que ocurre lo siguiente:
+ para la foto de la izquierda nos da una 'raw score' de 0.9, as? que predice correctamente que es la imagen de une gate,
+ para la foto del centro obtenemos una 'raw score' de 0.5, y a la vista de la imagen _interpretamos_ que a la red le cost? darnos una clasificaci?n correcta (nos la di? por los pelos) y que _obviamente_ no est? _tan segura_ porque le perre de la imagen parece un poco une gate, pero...

?y si le pedimos que clasifique la figura de la derecha? Cualquier cosa puede pasar, nos puede dar una 'raw score' por encima o por debajo de 0.5 o exactamente 0.5, pero en cualquier caso, sabemos que no es fiable. Y la cuesti?n es que **el modelo de aprendizaje autom?tico no nos podr?a decir 'hey, no tengo ni idea, s?lo he visto fotos de perres y gates' porque las 'raw scores' no muestran certidumbre**, a pesar de lo que todes hemos err?neamente interpretado alguna vez porque nos dijeron que 0.9 significa 90% de probabilidad de que sea gate, pero no es as? (ver secci?n <a href=#project-description-calibracion> 1.3 Calibraci?n </a> m?s adelante).  

![tweet_uq](./images/tweet_uq.png "tweet_uq")

En una tarea de regresi?n ocurre lo mismo. Entrenamos el modelo con un gran conjunto de datos de precios de casas y al aplicarlo sobre una casa nos da c?mo resultado, por ejemplo, 135.896,73 euros. La probabilidad de que el precio verdadero de la casa sea ese es nula. Si el modelo rinde bien, el precio verdadero estar? cerca de ese valor, estar? dentro de un intervalo de predicci?n alrededor de ese valor. Cuanto m?s ancho el intervalo, m?s grande la incertidumbre.

Y es importante recordar que los modelos de aprendizaje autom?tico supervisado est?n dise?ados para interpolar, **no para extrapolar, ni para identificar relaciones de causa-efecto, ni para abstraer conocimiento,... por lo cual les cuesta generalizar y pueden caer en el atajo de basar sus predicciones en correlaciones esp?reas y sin sentido**. Si no damos al modelo la oportunidad de avisarnos de que no est? preparado para desempe?ar la tarea, nos dar? igualmente una predicci?n, y tomaremos que est? igual de seguro de esa predicci?n como lo est? de la predicci?n de una instancia est?ndar para la que s? est? entrenado. Ser?a mucho m?s adecuado que el modelo de aprendizaje autom?tico nos informara de la certeza de sus resultados.  

Saber qu? no sabemos, i.e, cuantificar la incertidumbre, mejora la calidad de las decisiones para poder descartar las predicciones con alta incertidumbre y derivar las decisiones a expertes humanos, como ocurre en el denominado ['Active Learning'](https://en.wikipedia.org/wiki/Active_learning_(machine_learning)). Como ejemplo, el v?deo del art?culo [Robots That Ask For Help](https://robot-help.github.io/) de les desarrolladores del Large Language Model de Google, donde cuantifican incertidumbre con Predicci?n Conforme, el m?todo que explicamos en la secci?n <a href=#app-uq>4. Cuantificacion de Incertidumbre</a><br>.

<font color=#6B8E23 size="3">Ejemplos mostrando que las m?tricas de rendimiento como "accurary" no son suficientes y no indican certeza</font>

+ [M?trica incorrecta](https://koaning.io/posts/goodheart-bad-metric/), sobre la [ley de Goodhart](https://es.wikipedia.org/wiki/Ley_de_Goodhart): "cuando la m?trica se convierte el objetivo a optimizar, pierde su valor"
+ [La optimizaci?n m?trica no es suficiente](https://koaning.io/posts/high-on-probability-low-on-certainty/),
+ [Mean Squared Terror](https://koaning.io/posts/mean-squared-terror/),
+ [Accuracy as a Failure](https://www.infoq.com/presentations/project-failure-data-accuracy/),
+ [How to calibrate your classifier in an intelligent way using Machine Learning Conformal Prediction](https://valeman.medium.com/how-to-calibrate-your-classifier-in-an-intelligent-way-a996a2faf718),
+ [Correlaciones esp?reas divertidas](http://www.tylervigen.com/spurious-correlations).

#### <font color=#ac6240>1.1.1 Definici?n de Incertidumbre</font>

Seg?n la Wikipedia en castellano (consultada el 26.09.2022):

![uncertainty_wp](./images/uncertainty_wp.png "uncertainty_wp")

[Fuente de la imagen](https://es.wikipedia.org/wiki/Incertidumbre)

Vamos, que el asunto va de lo que pasa cuando queremos medir u observar algo o predecir cu?nto valdr? algo. 

Consultemos ahora a les expertes en [metrolog?a](https://es.wikipedia.org/wiki/Metrolog%C3%ADa), es decir, la Oficina Internacional de Pesas y Medidas ([BIPM](https://www.bipm.org/en/committees/jc/jcgm/), por sus siglas en franc?s) cuyo Joint Committee for Guides in Metrology (JCGM) publica la Gu?a para la expresi?n de la Incertidumbre en la Medida (GUM, por sus siglas en ingl?s) que incluye el International Vocabulary of basic and general terms in Metrology ([VIM](https://www.bipm.org/en/committees/jc/jcgm/publications) JCGM_100_2008_E-1)

<font size="5">????</font>


donde se define:

<blockquote>

**2.2 The term 'uncertainty'**

[...] Uncertainty (of measurement) [is a] parameter, associated with the result of a measurement, that characterizes the dispersion of the values that could reasonably be attributed to the measurand. [...] The parameter may be, for example, a standard deviation (or a given multiple of it), or the half-width of an interval having a stated level of confidence.
</blockquote>

y tambi?n:

<blockquote>
    
**3.3. Uncertainty**

The uncertainty of the result of a measurement reflects the lack of exact knowledge of the value of the measurand.
</blockquote>

Antes de ver qu? significa esto en el contexto del aprendizaje autom?tico, hagamos una aclaraci?n importante para distinguir bien entre los conceptos de incentidumbre, error, residuo y sesgo o 'bias'.

#### <font color=#ac6240>1.1.2 Incertidumbre vs error vs residuo vs bias</font>

El **error** es la diferencia entre el valor verdadero, (asumiento que existe y se puede definir, ojo, que aqu? esto es clave), y el valor resultado de la medici?n o predicci?n, por ejemplo, realizada emp?ricamente con un aparato de medida o en nuestro caso, el dato de si le paciente sufre o no fallo card?aco. 

Si ese valor verdadero lo defimos por un modelo, esa diferencia entre la medici?n y el modelo tambi?n se llama **residuo**, por ejemplo, la distancia entre un punto `(x,y)` (el dato resultado de la medici?n) y la recta de regresi?n lineal que intenta representar, es decir, modelar la realidad, que representan esos los datos, `y = mx + n`, y que idealizamos como representaci?n de la realidad. Vimos la regresi?n lineal brevemente en la seccion 4.1 Modelo de referencia inicial para la selecci?n de la m?trica: Regresi?n Log?stica del Jupyter Notebook `Exploration_and_Classification.ipynb`.

Entonces, en la representacion tradicional, la desviaci?n entre el valor obtenido de la medici?n y el ansiado supuesto valor verdadero se compone de dos tipos de errores:
+ el error aleatorio, la componente debida a la naturaleza [estoc?stica](https://en.wikipedia.org/wiki/Stochastic_process#Examples) del sistema, y
+ el error sistem?tico, tambi?n llamado **sesgo o 'bias'**, que es la componente reducible porque su media no es cero y se puede compensar con un 'factor de correccion' (mencionamos los problemas ?ticos del sesgo en IA en la intro del Jupyter Notebook `XAI.ipynb`).

En la Introducci?n del nuevo informe de 2012 [VIM 200:2012](https://www.bipm.org/en/committees/jc/jcgm/publications) se indica que estos dos tipos de errores no siempre son f?ciles de diferenciar y que no disponemos de una regla general para ver c?mo se combinan. Y tambi?n dice que normalmente estimamos el l?mite superior del valor absoluto del error total y que llamamos vagamente a eso 'incertidumbre'.

Pero la VIM tambi?n nos indica que es mejor no seguir ese punto de vista tradicional del 'error' porque:

+ necesita de la asumpci?n de la existencia un 'valor verdadero', y 
+ su objetivo es minimizar cualquier diferencia entre el resultado de la medida y ese supuesto 'valor verdadero', porque considera el error como una imperfecci?n indeseable,

siendo m?s adecuado tratar la dispersi?n de los resultados de las mediciones como una 'incertidumbre' que:

+ aporta valiosa informaci?n y detalles significativos del comportamiento del sistema, y 
+ nos permite asumir que **la medici?n s?lo se puede expresar como un intervalo de valores razonables**.

<font size="5"> ?? </font> 

Veamos ahora a qu? nos referimos con "intervalo de valores razonable".

#### <font color=#ac6240>1.1.3 Intervalo de valores razonables</font>

Hay muchas maneras de definir qu? entendemos por un 'intervalo de valores razonables' que nos permita cuantificar la posible variabilidad de los resultados y por tanto, la incertidumbre de la medici?n.

Los m?todos de UQ que aqu? explicamos nos proveen de un [intervalo de predicci?n](https://en.wikipedia.org/wiki/Prediction_interval) para cada predicci?n, es decir, es una estimaci?n del rango de valores que con la probabilidad que determinamos con el llamado 'nivel de confianza' (por ejemplo, con 0.90 tendremos el 90% de probabilidad) incluir? el valor correcto. 

Cuanto m?s estrecho es el intervalo de predicci?n, m?s podemos confiar en la predicci?n. Por ejemplo, data una predicci?n, `y_pred = 100`, calculamos su intervalo de predicci?n para que contenga al valor verdadero con un 90% de probabilidad, es decir, de cada 100 intervalos que construyeramos, 90 contendr?an a las "ground truth". Y la anchura del intervalo nos indica la confianza: si obtenemos el intervalo [99, 101] confiamos en el resultado m?s que si obtenemos el intervalo [50, 120].

###### Diferencias entre intervalos
[Intervalos de predicci?n](https://en.wikipedia.org/wiki/Prediction_interval): los que calculamos con Predicci?n Conforme, el m?todo que explicamos en la secci?n <a href=#app-uq>4. Aplicaci?n de m?todos de cuantificacion de incertidumbre</a>, con garant?a estad?stica contienen a la "ground truth", i.e., cumplen "validity", tambi?n llamada "garant?a de cobertura". Si la tarea es de clasificaci?n, contienen las clases en lugar de valores continuos y se llaman conjuntos de predicci?n e incluyen la "true label".

[Intervalos de confianza](https://es.wikipedia.org/wiki/Intervalo_de_confianza) de la estad?stica frecuentista: asociados a la incertidumbre al estimar un par?metro de la poblaci?n (por ejemplo, el valor esperado) a partir de un muestreo (por ejemplo, la media). Indican tambi?n un rango pero en ese caso indica una dispersi?n alrededor del par?metro estimador, no la incertidumbre de una predicci?n individual. El ejemplo cl?sico es el intervalo de confianza calculado con la desviaci?n est?ndar alrededor de la media de varias predicciones: `[y_mean - y_std, y_mean + y_std]`. No cumplen "validity".

[Intervalos de credibilidad](https://es.wikipedia.org/wiki/Intervalo_cre%C3%ADble) de los posteriores en la estad?stica bayesiana: incluyen la predicci?n con una probabilidad determinada, es decir, tratan sus l?mites como fijos y la predicci?n como una variable aleatoria (los intervalos de confianza frecuentista tratan sus l?mites como variables aleatorias y la predicci?n como un valor fijo).  Requieren el conocimiento de la distribuci?n a priori que es espec?fica de los inputs de cada situaci?n (algo para nada sencillo). No cumplen "validity".

Aqu? nos centramos en c?mo calcular intervalos de predicci?n para las predicciones arrojadas por modelos de aprendizaje autom?tico.

#### <font color=#ac6240>1.1.4 Incertidumbre en aprendizaje autom?tico</font>

Como dec?amos, cuando aplicamos un modelo de aprendizaje autom?tico es muy desable que no s?lo nos ofrezca una "raw score" (por ejemplo, la puntuaci?n cruda que sale de la ?ltima capa de activaci?n de una red neuronal) sino que nos diga adem?s cu?ndo no est? seguro y cu?nta incertidumbre est? asociada a su predicci?n. 

Podriamos incluso intentar desglosar los tipos de incertidumbre seg?n su origen. Por ejemplo, puede haber incertidumbre:
+ en c?mo se tomaron y procesaron los datos que usamos, la precisi?n de los aparatos de observaci?n y medida, el etiquetado, el almacenamiento, el ruido externo que hubo durante el proceso de recolecci?n de esos datos, ... 
+ en el elecci?n del modelo: dise?o, estimaci?n de par?metros, tipo de regularizacion, tipo de optimizaci?n, selecci?n de m?tricas, ..., 
+ en el error humano, el desconocimiento, los [sesgos cognitivos](https://www.visualcapitalist.com/50-cognitive-biases-in-the-modern-world/),...

y un mill?n de factores m?s.

Las fuentes de incertidumbre es un tema muy importantes pero demasiado complicado para un taller general como este porque requiere 'domain expertise' para cada caso. Dejamos un par de referencias en la bibliograf?a aqu? abajo. Aqu? nos centramos en la incertidumbre en la predicci?n y simplemente mencionaremos que en aprendizaje autom?tico se suelen considerar [dos tipos de incertidumbre](https://link.springer.com/article/10.1007/s10994-021-05946-3) en la predicci?n que se pueden resumir en:

+ **Aleatoria**: "?saldr? cara o cruz si lanzo una moneda al aire?", incertidumbre relativa al dato individual, **que no es reducible**, ni aunque aumentamos el n?mero de datos con el que entrenamos el modelo de aprendizaje autom?tico, y

+ **Epist?mica**: "?me f?o de que esta moneda no est? trucada?", incertidumbre relativa a la certeza del modelo de aprendizaje autom?tico despu?s de haber visto la instancia cuya 'target' ha de predecir, **s? se puede reducir** con m?s datos de entrenamiento.

En general, la incertidumbre aleatoria est? relacionada con lo que llamamos "varianza natural del fen?meno" o estocasticidad, que es la imposibilidad de conocer y controlar todos los factores e interacciones f?sicas y naturales de ese fen?meno en el mundo real. Porque si conoci?ramos todos los factores f?sicos de ese lanzamiento (peso, forma, y posici?n exacta de la moneda, ?ngulo y fuerza de lanzamiento, ecuaciones din?micas, velocidad del viento, densidad del aire, ...), es decir, si fu?semos el [Demonio de Laplace](https://es.wikipedia.org/wiki/Demonio_de_Laplace), podr?amos predecir con absoluta certeza si resultado es cara o cruz. Ser?a una falta de [conocimiento o episteme](https://dle.rae.es/episteme), as? que tendr?amos que atribuir parte de la incertidumbre aleatoria a la incertidumbre epist?mica y las barreras entre ambas se diluyen.

<font size="5"> ???? </font>

Pero la cosa se complica a?n m?s porque en realidad hay un tercer tipo de incertidumbre a tener en cuenta en aprendizaje autom?tico y que est? relacionado con el [sesgo inductivo](https://es.wikipedia.org/wiki/Sesgo), es decir, c?mo trata el algoritmo aquellas preguntas para las que no ha sido entrenado:

+ **'Drift'** tambi?n conocido como **'Shift'**: "?sigo a?n lanzando la misma moneda?", incertidumbre relacionada con los cambios en la distribuci?n de datos, cuando los nuevos datos no tienen la misma distribuci?n que tenian los datos de entrenamiento y estos nuevos datos pueden seguir teniendo o no la relaci?n que ten?an los datos de entrenamiento con la variable destino (ver [Concept Drift](https://en.wikipedia.org/wiki/Concept_drift) y tipos de [Data shift](https://www.nannyml.com/blog/types-of-data-shift), la librer?a [NannyML](https://github.com/NannyML/nannyml) sirve para detectarlo).
  

Ejemplos de 'drift' o 'shift' son cambios en la resoluci?n de la camara que hace las fotos para la clasificaci?n de imagenes, la persona que etiqueta los datos para una clasificaci?n, los gustos de les clientes para un 'sentiment analysis', ... o incluso la aparici?n de nuevos fen?menos, pensemos en la dificultad en la predicci?n de los efectos de cambio clim?tico ya que los modelos de aprendizaje autom?tico se basan en datos hist?ricos recogidos cuando las temperaturas globales no eran tan altas. 

En otras palabras, los modelos no pueden conocer m?s all? de los datos con los que han sido entrenados. Christoph Molnar pone un buen ejemplo [de sesgo inductivo en este post](https://www.linkedin.com/posts/christoph-molnar_youve-trained-a-machine-learning-model-activity-7186344006220468225-4__n/).

![comic_uq](./images/comic_uq.png "comic_uq")

[Fuente de la Imagen](https://www.smbc-comics.com/comic/rise-of-the-machines)

<font color=#6B8E23 size="4"> Conclusi?n</font>

+ Las decisiones basadas en predicciones deben tener en cuenta la incertidumbre. Por ejemplo, dada una predicci?n `y = 100`, es imprescindible saber si el intervalo de predicci?n es [99, 101] o [50, 120] para poder planear mejor los posibles escenarios, sobre todo al automatizar procesos sensibles o de riesgo.

+ Sin cuantificaci?n de la incertidumbre las buenas y las malas predicciones parecen iguales. La cuantificaci?n de la incertidumbre ofrece a los modelos de aprendizaje autom?tico la posibilidad de expresarse mejor.

<font color=#6B8E23 size="4"> Material adicional y referencias</font>

<font color=#6B8E23 size="3">Ejemplos de los peligros de la 'single-point prediction'</font>

+ La espectacular charla del gran Vincent Warmerdamm ['How to constrain artificial stupidity'](https://youtu.be/Z8MEFI7ZJlA).

+ El libro _'Weapons of math destruction'_ de Cathy O'Neil, hay [versi?n en castellano](https://www.indienauta.com/armas-destruccion-matematica-cathy-oneil-capitan-swing-2018/).

+ El libro ['Algorithms of Oppression'](https://en.wikipedia.org/wiki/Algorithms_of_Oppression) de Safiya Noble.

<font color=#6B8E23 size="3">Conceptos de UQ</font>

+ Sobre los tipos de incertidumbre en aprendizaje autom?tico:[_"Aleatoric and epistemic uncertainty in machine learning: an introduction to concepts and methods_"](https://link.springer.com/article/10.1007/s10994-021-05946-3).

+ Sobre las fuentes de incertidumbre en aprendizaje autom?tico, el cap?tulo "Uncertainty" del libro [_"Supervised Machine Learning for Science"_](https://ml-science-book.com/uncertainty.html) y el post de Christoph Molnar [_"A Pragmatic View of Uncertainty in Machine Learning"_](https://mindfulmodeler.substack.com/p/a-pragmatic-view-on-uncertainty-in) sobre este paper [_"Sources of Uncertainty in Machine Learning - A Statisticians' View"_](https://arxiv.org/abs/2305.16703).

+ Sobre los diferentes m?todos de cuantificaci?n de incertidumbre en aprendizaje autom?tico: [_"A Structured Review of Literature on Uncertainty in Machine Learning & Deep Learning_"](https://arxiv.org/abs/2406.00332).

+ En este blog, el art?culo introductorio: [_"A Comprehensive Introduction to Uncertainty in Machine Learning"_](https://imerit.net/blog/a-comprehensive-introduction-to-uncertainty-in-machine-learning-all-una/#:~:text=One%20way%20to%20estimate%20aleatoric,create%20a%20subset%20of%20samples), muy bueno, pero por desgracia no incluye los 'Conformal Predictors', nuestra herramienta favorita porque es la ?nica que garantiza la cobertura, como explicamos m?s abajo, secci?n <a href=#uq-cp>4.1 Conformal Predictors</a>.
 
+ La referencia por definici?n, el libro gratuito [_"Probabilistic Machine Learning: Advanced Topics"_](https://probml.github.io/pml-book/book2.html), desde 2022 por fin incluye los 'Conformal Predictors', a?n no le? el libro completo pero tengo que hacerlo.

<font size="5"> ?? </font>

Continuamos con la preparaci?n de los datos y el entrenamiento del modelo de aprendizaje autom?tico. Si ya trabajaste con los dos Jupyter Notebooks anteriores de este taller y est?s familiarizade con los datos y el modelo del bosque aleatorio, puedes pasar directamente a las secci?n  <a href=#project-description-calibracion> 1.3 Calibraci?n </a> y luego a la secci?n <a href=#app-uq> 4. Cuantificaci?n de incertidumbre</a>.

<a name='project-description-goal'/>

### <font color=#ac6240>1.2 Objetivo</font>

En este Jupyter Notebook vamos a aprlicar UQ a los resultados del modelo de bosque aleatorio (que no es interpretable) que entrenamos en el Jupyter Notebook `Exploration_and_Classification.ipynb`. No incluimos explicaciones exhaustivas y demostraciones matem?ticas (hay muchas otras fuentes, ver la lista de materiales arriba), mejor mencionaremos algunas caracter?sticas intuitivas sobre la cuantificaci?n de la incertidumbre y nos enfocaremos en su implementaci?n en Scikit-learn y software compatible.

Como dijimos, si ya conoces alguno de los dos Jupyter Notebook anteriores ya estar?s familiarize con los datos y con el bosque aleatorio, puedes pasar directamente a la secci?n <a href=#app-uq> 4. Cuantificaci?n de Incertidumbre</a>.

<a name='project-description-data'/>

### <font color=#ac6240>1.3 Datos y modelo de aprendizaje autom?tico</font>

Utilizaremos los datos y el modelo de aprendizaje autom?tico del Jupyter Notebook `Exploration_and_Classification.ipynb` que se encuentra en [este repositorio](https://github.com/MMdeCastro/Uncertainty_Quantification_XAI).

**Datos**

Utilizaremos los datos del [Conjunto de datos de predicci?n de insuficiencia card?aca de Kaggle](https://www.kaggle.com/fedesoriano/heart-failure-prediction) (cita: fedesoriano. (Septiembre de 2021) Heart Failure Prediction Dataset), y

Recordamos que las 'features' o caracter?sticas son los datos de lxs paciente ('Age', 'Cholesterol',...) y la variable objetivo o 'target', es decir, lo que queremos predecir, se llama "HeartDisease", y es una variable binaria que toma los valores:
+ 1: s?, pacientes que s? presentan riesgo de padecer una cardiopat?a
+ 0: no, pacientes que no presentan riesgo de padecer una cardiopat?a

En relacion a la 'feature' 'Sex', vamos a interpretar que se refiere al sexo con el que nacieron esas personas. Por otro lado vemos que s?lo toma los valores 'M' (male) or 'F' (female), es decir, entendemos que no hay datos de personas intersexuales o no binarias.

Por cierto que hemos elegido trabajar sobre un conjunto de datos tabulares porque los modelos de aprendizaje autom?tico y los m?todos de XAI y UQ suelen ser m?s f?ciles de entender si podemos ver los datos en columnas. Sin embargo los conceptos, tanto de XAI del notebook anterior, como de UQ que explicamos aqu?, sirven para cualquier aplicaci?n mientras sea de aprendizaje autom?tico. 

**Modelo de aprendizaje autom?tico para la clasificaci?n**

El modelo de clasificaci?n que utilizaremos es un bosque aleatorio que ya ajustamos, entrenamos, y validamos en el Jupyter Notebook anterior `Exploration_and_Classification.ipynb`. En este Jupyter Notebook directamente dividiremos el conjunto de datos en datos de entrenamiento y datos de prueba ('test'). Sobre esos datos de prueba aplicaremos el modelo de aprendizaje autom?tico en la secci?n <a href=#model-ml>3. Aplicaci?n del modelo de aprendizaje autom?tico: Bosque Aleatorio</a>.

Volvemos a remarcar que nuestro modelo s?lo es bueno encontrando correlaciones, (ni generaliza, ni entiende de relaciones de causa-efecto, ni conoce las restricciones del problema,...) por eso es fundamental que las decisiones finales sean tomadas por personas expertas en medicina. 

![ivory](./images/ivory_small.png "ivory")

[Fuente de la imagen: "Mean-squared-terror"](https://koaning.io/posts/mean-squared-terror/)

Y como es claramente una decisi?n de vida o muerte y como el bosque aleatorio no es un modelo interpretable, adem?s de aplicar m?todos de XAI que explicamos en el Jupyter Notebook anterior, aplicaremos aqu? m?todos de UQ para dotar de confianza a nuestros resultados y que nos indique como de seguro est? el modelo de aprendizaje autom?tico de su predicci?n. Lo haremos en la siguiente secci?n <a href=#app-uq>4. Aplicaci?n de m?todos de cuantificaci?n de incertidumbre.</a> 

Pero antes abordaremos un concepto fundamental, el de la calibraci?n, muy relacionado con el concepto de UQ.

<a name='project-description-calibracion'/>

### <font color=#ac6240>1.4 Calibraci?n</font>

Para cada nueve paciente, al aplicar `.predict_proba()` al modelo supervisado del bosque aleatorio anteriormente entrenado nos da una puntuaci?n o 'raw score' que si esta cerca de '0' clasifica a ese paciente en el grupo de pacientes sin riesgo a sufrir fallo card?aco y si esta cerca de '1' clasifica a ese paciente en el grupo de pacientes sin riesgo a sufrir fallo card?aco (donde 'cerca' se define por el 'umbral de decisi?n' como explicamos en el anterior Jupyter Notebook `Exploration_and_Classification.ipynb`, secci?n 4. Entrenamiento de los modelos). 

Como mencionamos en la 'Observaci?n importante' de esa misma secci?n 4. Entrenamiento de los modelos, las librer?as m?s populares de Python (Scikit-learn, XGBoost, Catboost, ... ) p?ntuan las prediciones con el m?todo `.predict_proba()` que, a pesar de su nombre, en realidad no produce probabilidades (ver por ejemplo, el art?culo [_"Are you sure that's a probability?"_](https://kiwidamien.github.io/are-you-sure-thats-a-probability.html)). En Keras `.predict_proba()` qued? obsoleto desde TensorFlow 2.6, hay que usar `.predict()`, en Torch ser?a por ejemplo, `torch.sigmoid()` para nuestro caso binario. Porque lo que ocurre es que se aplica una sigmoid (o una softmax para una tarea multiclase) que convierte el resultado del modelo, una variable continua, en una variable categ?rica, la clase. Es decir, en ese ?tlimo paso forzamos a un modelo "regresor" a "discriminar", es decir, a separar en clases.

![class_is_bad_reg](./images/class_is_bad_reg.png "class_is_bad_reg")

[Fuente de la imagen](https://www.linkedin.com/posts/carl-mcbride-ellis_machinelearning-activity-7296431605462495232-JTvO?utm_source=share&utm_medium=member_desktop&rcm=ACoAACIj_TMBx_CF4v3uJ-6ZWtbNUYgjFblhsno)

Y es muy com?n ver c?mo en muchos libros, cursos, v?deos,... se toma esa 'raw score' discretizada como una probabilidad, en nuestro caso la probabilidad de que ese nueve paciente est? o no en riesgo de sufrir una insuficiencia card?aca. Como veremos en la secci?n <a href=#app-uq>4. Cuantificaci?n de Incertidumbre</a>, estas puntuaciones son s?lo heur?sticas, no est?n calibradas, es decir, el modelo no ha tenido en cuenta cu?ntas peronsas ten?an o no fallo card?aco y no son por tanto verdaderas probabilidades. Sobre c?mo los modelos de aprendizaje autom?tico no necesariamente est?n calibrados por defecto, ver por ejemplo estos art?culos: 
+ para Naive Bayes, Decision Trees, y Neural Networks [_?Targeting the uncertainty of predictions at patient-level using an ensemble of classifiers coupled with calibration methods, Venn-ABERS, and Conformal Predictors?_](https://pubmed.ncbi.nlm.nih.gov/31816401/) 

+ para Boosting Trees, Random Forest, Neural Networks, K Nearest Neighbours, Naive Bayes and Support Vector Machine: [_?Predicting good probabilities with supervised learning?_](https://dl.acm.org/doi/10.1145/1102351.1102430)

+ para Redes Neuronales: [_?On Calibration of Modern Neural Networks?](https://proceedings.mlr.press/v70/guo17a/guo17a.pdf) y [_?A survey of uncertainty in deep neural networks?_](https://link.springer.com/article/10.1007/s10462-023-10562-9)

+ para Regresi?n Log?stica: [_?Don't Just Blame Over-parametrization for Over-confidence: Theoretical Analysis of Calibration in Binary Classification?_](https://arxiv.org/abs/2102.07856).

Para profundizar m?s sobre la calibraci?n en general, este reciente estudio es muy completo: [_A Benchmark Study on Calibration_](https://www.taolinwei.com/calibration-study/)

Cuando nos centramos en predecir probabilidades en lugar de clases, eliminamos la necesidad de considerar un umbral de decisi?n y por tanto, se resuelve el asunto de los data set no balanceados, ver por ejemplo [_"Classification vs. Prediction"_](https://www.fharrell.com/post/classification/):

    [...] that classifiers should not be used at all when there is little variation in the outcome variable (prevalence is near 0 or 1), and that only tendencies (probabilities) should be modeled.

Primero vamos a seguir el flujo de operaciones est?ndard i.e., predecir las clases sin cuantificar la incertidumbre a?n, y una vez obtenidas esas "raw scores", si el resultado de aplicar el modelo de aprendizaje autom?tico sobre los datos de une paciente nos arroja una puntuaci?n o 'raw score' de 0.7, si el umbral de decisi?n es igual a 0.4, le paciente quedar? clasificade como 'perteniente al grupo en riesgo de sufrir fallo card?aco'. 

Luego mediremos cu?nto de calibrado est? el modelo, es decir, si las puntuaciones crudas que nos arroja son verdaderas probabilidades. Aplicaremos las m?tricas de calibraci?n m?s comunes, la Log Loss (tambi?n conocida como Cross-Entropy) y Brier Score.

Por ?ltimo, en la secci?n <a href=#app-uq>4. Cuantificaci?n de Incertidumbre</a> aplicaremos Predicci?n Conforme, que nos permite calibrar los resultados para obtener verdaderas probabilidades para cada clase y adem?s nos proporciona intervalos de predicci?n para evaluar la incerticumbre de dichos resultados. Como cualquier otro m?todo de calibraci?n, la Predicci?n Conforme requiere que reservemos una parte de los datos que el modelo no habr? visto durante el entrenamiento, ver m?s abajo la secci?n <a href=#prep-split>2.2.Divisi?n de los datos en conjuntos de datos de entrenamiento, calibraci?n y prueba.<a/>

<a name='project-description-software'/>

### <font color=#ac6240>1.5 Software</font>

Usaremos estas librer?as:

In [ ]:
import pandas as pd     
import numpy as np

# data exploration and preparation  
from sklearn.metrics import mutual_info_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer 
from sklearn.preprocessing import StandardScaler 

# model
from sklearn.ensemble import RandomForestClassifier

# performance metrics
from sklearn.metrics import accuracy_score, f1_score, auc, recall_score, precision_score
from sklearn.metrics import make_scorer, confusion_matrix
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score

# calibration metrics
from sklearn.metrics import log_loss, brier_score_loss


# plotting and displaying in the notebook
import seaborn as sns
from matplotlib import pyplot as plt
#from IPython.display import display
#import emoji

# ignore warnings
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

<a href=#toc>Subir a la Tabla de Contenidos</a><br>

<a name='prep'/>

## <font color=#ac6240>2. Preparaci?n de los datos</font>

La exploraci?n y preparaci?n de los datos se hizo en el Jupyter Notebook `Exploration_and_Classification.ipynb` que se encuentra en [este repositorio](https://github.com/MMdeCastro/Uncertainty_Quantification_XAI), aqu? s?lo mostarmos los pasos esenciales.

<a name='prep-clean'/>

### <font color=#ac6240>2.1 Limpieza de los datos</font>

Cargamos los datos del fichero que esta en el mismo repositorio:

In [ ]:
path = 'data/'  
df = pd.read_csv(path + "heart_failure.csv")
df.head()

'FastingBS' (la presi?n sangu?nea en ayunas) deber?a ser una variable categ?rica ya que los valores actuales de '0' y '1' son solamente representaciones de valores cualitativos que no queremos que el modelo interprete como valores cuantitativos.

In [ ]:
# Transform the categorical values wrongly classified as numericals
new_values = {
    0: 'L', # low
    1: 'H', # high
}

df.FastingBS = df.FastingBS.map(new_values)
# type(df.FastingBS[0]) # uncomment to check that it is a string type

In [ ]:
# Store in different lists the names of the categorical and numerical columns since they will be treated different 
categorical = df.select_dtypes(include=['object']).columns.tolist()  # for strings 
numerical = df.select_dtypes(include=['int64','float64']).columns.tolist() # for numbers

Los valores m?nimos de 'RestingBP' y 'Cholesterol' son cero, lo cual es sospechoso. Con experiencia en el campo de la medicina, podr?amos decidir si reemplazar estos valores nulos por la media u otro valor. Carecemos de esta informaci?n, por lo tanto, simplemente eliminamos esas filas.

In [ ]:
index_to_drop = list(df.loc[(df.RestingBP == 0) | (df.Cholesterol == 0)].index)
df.drop(index_to_drop, inplace=True)

<font size ='5'> ?? </font>No hay necesidad de m?s limpieza de datos.

<a name='prep-split'/>

### <font color=#ac6240>2.2 Divisi?n de los datos en conjuntos de datos de entrenamiento, calibraci?n y prueba</font>

En esta divisi?n aleatoria, y en todas las ocasiones siguientes en las que usemos un generador de n?meros aleatorios, fijamos la semilla del generador de n?meros aleatorios para asegurar la reproducibilidad de los resultados. Fijamos el `random_state = 1` pero podr?a ser 2 o cualquier otro valor, no afectar? a los resultados, pero con esta correcci?n siempre obtendremos exactamente los mismos resultados.

Recordamos que dividimos el conjunto de datos en entrenamiento, calibraci?n, y prueba porque la exploraci?n de los datos y el ajuste y la validaci?n del modelo de clasificaci?n ya lo hicimos en el Jupyter Notebook `Exploration_and_Classification.ipynb` que se encuentra en [este repositorio](https://github.com/MMdeCastro/Uncertainty_Quantification_XAI).

Para poder cuantificar la incertidumbre, necesitamos guardar unos datos que nos ayuden a calibrar el modelo, por eso hacemos una segunda partici?n del conjunto de entrenamiento y lo separamos en 'proper train' y 'cal'. Es decir, para aplicar cuantificaci?n de incertidumbre separamos tambi?n en 3 sets, pero el tercer conjunto lo usamos para calibrar, no para validar. Si se quiere validar, hay que separa en 4 sets. Aqu? separamos en 3 sets porque ya damos el modelo por validado.

In [ ]:
# copy the original dataframe so further manipulations will not affect it 
df_select = df.copy() 
categorical = df_select.select_dtypes(include=['object']).columns.tolist()
numerical = df_select.select_dtypes(include=['int64','float64']).columns.tolist()
numerical.remove('HeartDisease')

In [ ]:
numerical

In [ ]:
# first separate the train from the test set
df_train, df_test = train_test_split(df_select, test_size=0.04, random_state=1)
# separate the train set into calibration set and proper training set
df_proper_train, df_cal = train_test_split(df_train, test_size=0.5, random_state=1)

len(df_train), len(df_cal), len(df_test)

In [ ]:
# reset index
df_proper_train = df_proper_train.reset_index(drop=True)
# define the target
y_proper_train = df_proper_train.HeartDisease.values
# remove the target from the features
del df_proper_train['HeartDisease']

<a name='prep-encoding'/>       

### <font color=#ac6240>2.3 Codificaci?n y normalizaci?n de las 'features'</font>

Para codificar las caracter?sticas categ?ricas usamos Scikit-Learn `DictVectorizer`, que toma un diccionario y lo convierte en un vector, es decir, en un `numpy.array`. Es un m?todo de los llamados One-Hot Encoding (OHE) para convertir las caracter?sticas categ?ricas (no afectar?a las num?ricas si se lo aplicasemos) en columnas con valores binarios, con tantas columnas como valores tome la variable categ?rica. Por ejemplo, la columna de la caracterist?ca 'ExcerciseAngina' que pod?a tener valores 'N' o 'Y' ahora dar? lugar a dos columnas, una llamada 'ExcerciseAngina=N' y otra 'ExcerciseAngina=Y' que claramente son dependientes (dan la misma informaci?n) y est?n totalmente correlacionadas (cuando el valor de una es '0', el valor de la otra es seguro '1', ver las correlaciones en el Jupyter Notebook anterior). Cualquier m?todo de codificaci?n a?ade por tanto informaci?n redundante. Hemos de eliminar la informaci?n redundante desechando una de las columnas de cada par correlacionado (da igual qu? columna de cada par eliminemos, ambas dan la misma informaci?n), quitaremos 'ExerciseAngina=N', 'FastingBS=H', 'Sex=F', 'ST_Slope=Up':

In [ ]:
dv = DictVectorizer(sparse=False) # False bcs is not a sparse matrix (we do not have many zeros)

# TRAIN
proper_train_dict = df_proper_train[categorical].to_dict(orient='records') # records = to do it row-wise, not col-wise
X_proper_train_cat = dv.fit_transform(proper_train_dict) # make it a vector

df_X_proper_train_cat = pd.DataFrame(X_proper_train_cat,columns= dv.get_feature_names_out())
# remove redundant columns
df_X_proper_train_cat.drop(['ExerciseAngina=N', 'FastingBS=H', 'Sex=F', 'ST_Slope=Up'], 
                    axis=1, inplace=True)
# convert the dataframe to a np.array again
X_proper_train_cat = df_X_proper_train_cat.to_numpy()

categorical_update = df_X_proper_train_cat.columns.to_list()

En el Jupyter Notebook de la exploraci?n y clasificaci?n de los datos usamos Scikit-Learn `StandardScaler` para normalizar las caracter?sticas num?ricas entre 0 y 1. As?, aquellas columnas con valores en un rango m?s alto no tendr?an m?s representaci?n y modelos como la regresi?n log?stica converg?an mejor. Como aqu? usamos un bosque aleatorio, no necesitamos escalar las 'features' num?ricas. 

In [ ]:
#scaler = StandardScaler()

# TRAIN
X_proper_train_num = df_proper_train[numerical].values
#X__propertrain_num = scaler.fit_transform(X_proper_train_num)

Unimos las matrices num?ricas y categ?ricas:

In [ ]:
# TRAIN
X_proper_train = np.column_stack([X_proper_train_num, X_proper_train_cat])

Las nuevas caracter?sticas son:

In [ ]:
feature_names = numerical + categorical_update
feature_names

<font size="5"> ?? </font>  Ya tenemos todo para entrenar nuestro bosque aleatorio.

<a href=#toc>Subir a Tabla de Contenidos</a><br>

<a name='model-ml'/>

## <font color=#ac6240>3. Aplicaci?n del modelo de aprendizaje autom?tico: Bosque Aleatorio</font>

Entrenaremos el modelo de aprendizaje supervisado llamado bosque aleatorio ('Random Forest') que es uno de los clasificadores de aprendizaje autom?tico m?s populares (ver, por ejemplo, [Kaggle 2021](https://storage.googleapis.com/kaggle-media/surveys/Kaggle's%20State%20of%20Machine%20Learning%20and%20Data%20Science%202021.pdf) p?gina 32). En el Jupyter Notebook `Exploration_and_Classification.ipynb` que se encuentra en [este repositorio](https://github.com/MMdeCastro/Uncertainty_Quantification_XAI) probamos varios modelos de clasificaci?n y el bosque aleatorio result? ser el ganador.

El bosque aleatorio es un tipo de [Ensemble Learning](https://en.wikipedia.org/wiki/Ensemble_learning): corremos en paralelo muchos ?rboles de decisi?n y tomamos la predicci?n del que obtuvo m?s aciertos. Primero, se crean m?ltiples conjuntos de datos de entrenamiento mediante el muestreo de datos del conjunto de datos de entrenamiento (un m?todo conocido como 'bootstrapping' o 'bagging') y luego se entrena un ?rbol de decisi?n en cada uno de estos conjuntos de datos.  Pueden usarse para resolver problemas de regresi?n o de clasificaci?n. Por lo general, los bosques aleatorios tienen un alto rendimiento, pero no son interpretables (son 'black-boxes') y no son recomendables para grandes conjuntos de datos (~ TB) porque requieren mucha m?s potencia de computadora que la regresi?n log?stica, por ejemplo.

<a name='model-metric'/>

### <font color=#ac6240>3.1 Selecci?n de la M?trica y del umbral de decisi?n</font>

En el Jupyter Notebook `Exploration_and_Classification.ipynb` que se encuentra en [este repositorio](https://github.com/MMdeCastro/Uncertainty_Quantification_XAI), decidimos medir la calidad del modelo con la metrica denominada 'F1-score' porque **queremos evitar los Falsos Negativos (no queremos diagnostiar como sane a une paciente en riesgo)**. Adem?s, contaremos la tasa de Verdaderos Positivos (o 'recall') y no perderemos de vista la precisi?n ('precision') y exactitud ('accuracy').


Por ?ltimo, concluimos que defin?amos el umbral de decisi?n en `t = 0.4`.

<a name='model-RF'/>

### <font color=#ac6240>3.2 Entrenamiento, prueba, y evaluaci?n del bosque aleatorio</font>

No vamos a repetir todo el proceso de ajuste y validaci?n sino que directamente entrenamos y probamos ('test') el modelo de clasificaci?n del Jupyter Notebook `Exploration_and_Classification.ipynb` para poder comenzar la cuantificaci?n de la incertidumbre en el siguiente apartado. 

#### <font color=#ac6240>3.2.1 Entrenamiento y prueba del bosque aleatorio</font>

Entrenamos el modelo ganador con los datos de entrenamiento: 

In [ ]:
# train the machine learning model: random forest

RF = RandomForestClassifier(max_depth = 10, min_samples_leaf = 5, 
                            n_estimators =  50, random_state = 1)

model = RF.fit(X_proper_train, y_proper_train)

Vamos aplicarlo sobre los datos de prueba, primero los preparamos:

In [ ]:
# preparation TEST data

# reset index after the splitting shuffling
df_test = df_test.reset_index(drop=True)
# define target of the test set
y_test = df_test.HeartDisease.values
   
# 1. encode the categorical features and remove redundant columns-------------    

test_dict = df_test[categorical].to_dict(orient='records')
X_test_cat = dv.transform(test_dict) 

# convert the data array to a dataframe
df_X_test_cat = pd.DataFrame(X_test_cat,columns= dv.get_feature_names_out())

# remove redundant columns
df_X_test_cat.drop(['ExerciseAngina=N', 'FastingBS=H', 'Sex=F', 'ST_Slope=Up'], 
                    axis=1, inplace=True)

# convert the dataframe to a np.array again
X_test_cat = df_X_test_cat.to_numpy()

# 2. scale the numerical features --------------------------------------------

X_test_num = df_test[numerical].values
#X_test_num = scaler.transform(X_test_num) 

# 3. join the matrices -------------------------------------------------------

X_test = np.column_stack([X_test_num, X_test_cat]) 

#### <font color=#ac6240>3.2.1 Evaluaci?n del rendimiento del modelo clasificador sobre el conjunto de prueba</font>

In [ ]:
# model application on TEST data
y_pred = RF.predict_proba(X_test)[:, 1]

# performance metrics with decision threshold
t = 0.4

acc = accuracy_score(y_test, y_pred >= t) 
rec = recall_score(y_test, y_pred >= t)
f1  = f1_score(y_test, y_pred >= t) 
auc = roc_auc_score(y_test, y_pred)
print('For the test dataset:',
      'ACC:', acc.round(3), 'Recall:', rec.round(3),'F1:', f1.round(3),
      'ROC AUC:', auc.round(3))

In [ ]:
# confusion matrix

cf_matrix = confusion_matrix(y_test, y_pred >= t)

ax = sns.heatmap(cf_matrix, annot=True, cmap='Blues')

ax.set_title('Confusion Matrix\n');
ax.set_xlabel('\nPredicted Values')
ax.set_ylabel('Actual Values ');

ax.xaxis.set_ticklabels(['0','1'])
ax.yaxis.set_ticklabels(['0','1'])

##plt.figure(figsize=(7,5))
plt.rcParams['figure.figsize'] = [7, 5]  # re-run this cell to get the correct figure size

plt.show();

<font size="5"> ?? </font> Como es un contexto m?dico, queremos evitar los Falsos Negativos y hemos obtenido s?lo 2, con un porcentaje de aciertos de aproximadamente el 85% (ver los valores de las m?tricas obtenidos), no est? mal. Adem?s, vimos en el Jupyter Notebook anterior que estos hiperparametros nos dan valores de las m?tricas similares a los de los conjuntos de datos de entrenamiento y validaci?n, por lo que no estamos sobreajustadando ('overfitting') y esperamos que el modelo generalice bien dentro de lo posible.

#### <font color=#ac6240>3.2.2 Evaluaci?n de la calibraci?n del modelo clasificador sobre el conjunto de prueba</font>

Como dec?amos m?s arriba las predicciones de los modelos de aprendizaje autom?tico no est?n bien calibradas por defecto. Como veremos en la secci?n <a href=#app-uq>4. Cuantificaci?n de Incertidumbre</a> esto significa que una puntuaci?n igual a 0.2 no quiere decir que haya un 20% de probabilidad de ese sea el valor verdadero de esa predicci?n. En esa secci?n explicamos c?mo calibrar nuestras predicciones para nuestra clasificaci?n binaria.  

Aqu? comprobamos la calibraci?n de nuestro Bosque Aleatorio. Dada "y_pred_positive_class", i.e., la predicci?n cruda ("raw score") de la clase 1 ("positive class"),  las m?tricas m?s apropiadas ("proper scores") son:
+ Brier Score: (y_pred_positive_class - y_true_label)^2
+ Log-Loss (tambi?n conocida como logistic loss o cross-entropy loss): 
    + -log(y_pred_positive_class) para la clase 1, y 
    + -log(1-y_pred_positive_class) para la clase 0.
    
Ambas m?tricas indican mejor calibraci?n cuando su valor se acerca a cero y calibraci?n perfecta cuando valen cero, por eso se denominan ["Strictly Proper Score Rules"](https://www.tandfonline.com/doi/abs/10.1198/016214506000001437).

In [ ]:
metrics_list = []
l_loss = log_loss(y_test, y_pred)
brier_loss = brier_score_loss(y_test, y_pred)
metrics_list.append(["Random Forest", acc.round(3), f1.round(3), brier_loss.round(3), l_loss.round(3)])
metrics_list

Vemos que las m?tricas de rendimiento sobre el conjunto de prueba, exactitud y F1, rondan ambas el 80% y que las m?tricas de calibraci?n indican que el modelo clasificador no est? muy calibrado.

<a href=#toc>Subir a Tabla de Contenidos</a><br>

Empezamos ya a implementar UQ para incluir la certeza de la predicci?n en lugar de un ?nico valor puntual. Identificaremos as? cuando nuestro modelo no est? seguro de su predicci?n y podremos reducir el n?mero de Falsos Negativos.

![Molnar_UQ](./images/Molnar_UQ.png "Molnar_UQ")

[Fuente de la imagen](https://twitter.com/ChristophMolnar)

<a href=#toc>Subir a Tabla de Contenidos</a><br>

<a name='app-uq'/>

## <font color=#ac6240>4. Cuantificacion de incertidumbre</font>

Como mencionamos m?s arriba, las librer?as m?s populares de Python (Scikit-learn, XGBoost, ... ) p?ntuan las prediciones con el m?todo `.predict_proba()` que, a pesar de su nombre, en realidad no produce probabilidades (ver por ejemplo, el art?culo [_"Are you sure that's a probability?"_](https://kiwidamien.github.io/are-you-sure-thats-a-probability.html)). En Keras `.predict_proba()` qued? obsoleto desde TensorFlow 2.6, hay que usar `.predict()`, en Torch ser?a por ejemplo, `torch.sigmoid()` para nuestro caso binario. 

Estas puntuaciones crudas o 'raw scores' que salen de la ?ltima funci?n de activaci?n (normalmente una sigmoid o softmax, aunque en las capas internars usaras `activation='relu'` ver por ejemplo los [docs de Scikit-learn](https://github.com/scikit-learn/scikit-learn/blob/7b136e9/sklearn/neural_network/multilayer_perceptron.py#L267
)) van entre 0 y 1 y todas juntas suman 1 (por ejemplo, en una tarea de clasificaci?n binaria como la del ejemplo de la secci?n 1.1 <a href=#project-description-intro>Introducci?n</a>, si una instancia obtiene 0.9 para la clase "gate", entonces obtiene 0.1 para la clase "perre") pero no por eso son probabilidades porque **no est?n calibradas, es decir, no es necesariamente cierto que las instancias que obtuvieron 0.9 sean un 90% "gates"**. Como mucho podemos considerarlas s?lo como 'probabilidades heur?sticas' es decir, sin garant?a estad?stica. Asumir que las 'raw scores' son verdaderamente probabilidades causa consecuencias desastrosas a la hora de inferir informaci?n o al intentar sumar estas puntuaciones a verdaderas probabilidades, como se explica en [_"How to calibrate your classifier in an intelligent way using Machine Learning Conformal Prediction"_](https://valeman.medium.com/how-to-calibrate-your-classifier-in-an-intelligent-way-a996a2faf718).

Como dec?amos, las puntuaciones est?n calibradas para ser verdaderas probabilidades si, por ejemplo, entre todas las clasificaciones con una puntuaci?n del 90% encontramos la clase verdadera 9 de cada 10 veces. Por ejemplo, cuando en la secci?n 1.1 <a href=#project-description-intro>Introducci?n</a> habl?bamos de la clasificaci?n binaria de im?genes de "perres" (0) y "gates" (1) y la primera foto (que era une gate) obten?a una 'raw score' de 0.9, para que esa 'raw score' fuera una verdadera probabilidad deber?amos encontrar una foto de une gate el 90% de las veces que una foto obtuviera un 0.9 en la fase de inferencia. Para ver que as? sea, hemos de calibrar esa 'raw score' con el numero de im?genes de gates que realmente hab?a en el conjunto de datos de calibraci?n (algo que ni el m?todo `.predict_proba()` ni `.predict()` hacen). Lo que ocurre a menudo es que al calibrar esa 'raw score' obtenemos un nuevo valor. Por ejemplo, la verdadera puntuaci?n (que ya es una verdadera probabilidad) de la foto de ese gate puede que pase de 0.9 a 0.7.

Aqu? vamos a explicar la familia de m?todos denominados ['Conformal Predictors'](https://en.wikipedia.org/wiki/Conformal_prediction) los cuales funcionan para cualquier modelo de aprendizaje autom?tico y nos permiten calibrar nuestras 'raw scores' para que sean verdaderas probabilidades y en el proceso obtendremos los intervalos de predicci?n tal que: 

+ de la 'raw score' de 0.9 para el ejemplo de la foto de le gate, obtendr?amos algo como [0.75, 0.80], lo que nos da la idea de que el modelo de clasificaci?n est? bastante seguro de que es une gate, 
+ de la 'raw score' de 0.5 para el ejemplo de la foto de le perre con espuma en la cabeza, obtendr?amos algo como [0.35, 0.65], lo que nos dice que no nos fiemos de ese resultado, y
+ para la imagen de la p?jaro la 'raw score' calibrada ser?a algo como [0.1, 0.8], es decir, algo muy poco fiable que nos avisa que no debemos confiar en esa predicci?n. 

A continuaci?n aplicamos un ejemplo para nuestra predicci?n de fallo card?aco en la secci?n <a href=#uq-cp>4.1 Conformal Predictors</a>.

<a name='uq-cp'/>

### <font color=#ac6240>4.1 Conformal Predictors</font>

La ['Conformal Prediction'](https://en.wikipedia.org/wiki/Conformal_prediction) es una familia de m?todos para cuantificar la incertidumbre de los resultados de un modelo y saber as? cu?nto podemos confiar en ellos. Recibe su nombre de la denominada medida de no conformidad, o 'nonconformity measure' que cuantiza cu?n diferente es la  predicci?n dentro del conjunto de predicciones que obtuvimos al predecir el conjunto de entrenamiento. Una alta 'nonconformity' indica que el nuevo dato es raro, como la foto de le p?jare en el ejemplo de clasificaci?n de perres y gates de la Introducci?n de este Jupyter Notebook. Algunos autores y autoras la llaman medida de conformidad, es lo mismo, son maneras de hablar.

Las personas que aplicamos Conformal Predictions nos llamamos 'nonconformists'.


<font size="5">??<font/>
    
Los miembros de la familia de la Predicc?n Conforme son esencialmente calibradores de las predicciones que en el proceso de calibraci?n generan regiones de predicci?n alrededor de dichas predicciones. Cu?nto m?s estrecha es la regi?n, m?s confiamos en la predicci?n. Si la predicci?n puede tomar valores cont?nuos, como en tareas de regresi?n, las regiones son intervalos, por ejemplo [49,55] y si pueden tomar valores discretos, como ocurre en clasificaci?n, las regiones son conjuntos ("sets"), por ejemplo ["gate", "perre"].

La Predicci?n Conforme:

+ es 'model-agnostic', funciona para cualquier modelo de aprendizaje autom?tico y para cualquier tarea, ni siquiera necesita conocer al modelo de aprendizaje autom?tico para el que est? cuantizando la incertidumbre de sus predicciones,     
+ ofrece garant?a de cobertura demostrada matem?ticamente: los intervalos de predicci?n vienen con garant?as de contener el resultado verdadero dado un nivel de confianza determinada, por ejemplo, garantizan contener a la 'ground truth' el 90%, 95%, o 99% de las veces,
+ se aplica en el 'post-processing' sobre las predicciones del modelo de aprendizaje autom?tico ya entrenado (no hay que reentrenar el random forest, en este caso), es decir, una vez obtenidas las predicciones, las conformalizamos,
+ requiere pocas l?neas de c?digo y es r?pida, 
+ funciona para datos recogidos en tiempo real, es decir, 'online assimilation',
+ a diferencia de los m?todos param?tricos, no es necesario asumir que la variable de destino o 'target' est? distribuida seg?n una cierta distribuci?n (casi nada en el universo puede aproximarse a una distribuci?n normal y recordemos que el Teorema del L?mite Central no indica eso, lo que indica es que las _medias_ de las distribuciones se aproximan a una distribuci?n normal).

De hecho, la ?nica condici?n para garantizar la cobertura de los intervalos de predicci?n calculados con 'Conformal Prediction' es que las instancias del conjunto de calibraci?n y prueba sean [intercambiables](https://en.m.wikipedia.org/wiki/Exchangeable_random_variables), es decir, que el orden de los datos no importe, (como en nuestro caso, el clasificador Random Forest no requiere que les pacientes est?n ordenades, cualquier ordenaci?n de les pacientes arrojar? estad?sticamente el mismo resultado) lo cual es una restricci?n menos fuerte que la t?pica restricci?n sobre que los datos han de ser independientes e estar id?nticamente distributidos o [iid](https://es.wikipedia.org/wiki/Variables_aleatorias_independientes_e_id%C3%A9nticamente_distribuidas). Aunque desde 2021 ya hay algoritmos de CP para confomalizar series temporales (ver siguiente subsecci?n) que demuestran emp?ricamente poder generar intervalos de predicci?n que continene a la predicci?n verdadera (es decir, la demostraci?n es emp?rica, no matem?tica). Y un art?culo muy reciente ha demostrado que ni siquiera se necesita asumir intercambiabilidad, aunque el coste computacional de este m?todo es algo: ["_Conformal Validity Guarantees Exist for Any Data Distribution (and How to Find Them)"_](https://arxiv.org/abs/2405.06627). A continuaci?n aplicamos la "Inductive Conformal Prediction" o "Split Conformal Prediction" que no requiere reentrenar el modelo de aprendizaje autom?tico para cada inferencia, en contraposici?n de la "Transductive Conformal Prediction", la CP original, que implica reentrenar en cada predicci?n.

As?, la 'Conformal Prediction' nos cuantifica la incertidumbre en forma de intervalos de predicci?n, tambi?n llamados 'conformal intervals' (para tareas de clasificaci?n, 'conformal sets'), que con garant?a matem?tica incluyen la prediccion correcta con cierta probabilidad, lo cual se llama 'coverage guarantee', seg?n el nivel de confianza deseado. Entonces, en lugar de obtener predicciones puntuales ('single-point predictions'), es decir, un n?mero real en el caso de una regresi?n o la etiqueta de una clase (recordemos el ?ltimo ejemplo en `Exploration_and_classification.ipynb` donde la nueva paciente obtenia una 'raw score' de 0.32, con el umbral de decisi?n en 't = 0.4', nos clasificaba a esa paciente con una ?nica clase 'HeartDisease = 0') en el caso de una clasificaci?n, los resultados calibrados con 'Conformal prediction' nos dan un intervalo de n?meros reales o un conjunto de etiquetas, respectivamente.
    
Por ejemplo, aplicando 'Conformal Prediction' a los resultados de un modelo de aprendizaje autom?tico (una Deep Neural Network en PyTorch) para una clasificaci?n de im?genes, vemos que ?ste:
    
+ no est? muy seguro de si la primera foto era un osciloscopio, as? que nos lo ofrece como primera opci?n en un conjunto de predicci?n con otras dos opciones, 'microondas' y 'port?til', 
+ tampoco est? muy seguro de la foto central, de nuevo nos di? la respuesta correcta como primera opci?n pero el conjunto de predicci?n contiene tambi?n 'refrigerador' y ' lavadora', y
+ est? muy seguro de que en la foto de la derecha hay une perre de raza 'Irish Setter' (seguro que ha visto muchos como ese, para el modelo no es para nada una imagen rara), y es la ?nica opci?n del conjunto de predicci?n:

![CP_example](./images/CP_4.png "CP_example")

[Fuente de la imagen: Google Colab de A. Angelopoulos desarrollador de MAPIE](https://colab.research.google.com/github/aangelopoulos/conformal_classification/blob/master/example.ipynb#scrollTo=oIiBD-pFVdkD)

Recordamos que los intervalos o conjuntos de predicci?n que obtenemos con la 'Conformal Prediccion' no son un mero ranking de las 'raw scores' de cada clase si no una regi?n de predicci?n con garant?as estad?sticas de que el valor verdadero est? dentro de ella.

<font color=#6B8E23 size="4"> Material adicional y referencias</font>

<font color=#6B8E23 size="3"> Cursos, charlas, y blogs

+ Las slides de mi charla ["Conformal Prediction: Key Concepts"](https://docs.google.com/presentation/d/1bQYIFyQysQPx79wJq1mltylsH_aeO2AnLYRlHiLq6vo/edit#slide=id.g2dd387acb29_0_4)
 
+ Los [art?culos gratuitos de Medium de Valeriy Manokhin](https://valeman.medium.com/), miembro del grupo de investigaci?n que cre? la 'Conformal Prediction' y su [magn?fico repositorio](https://github.com/valeman/awesome-conformal-prediction) con materiales sobre CP.
    
+ Este art?culo de blog ["Conformal prediction theory explained"](https://medium.com/low-code-for-advanced-data-science/conformal-prediction-theory-explained-14a35226df80) de un miembro del equipo que desarrolla el paquete de Predicci?n Conforme CREPES explica la Predicci?n Conforme en general. 

+ El curso gratuito de Christoph Molnar (subscr?bete gratis [aqu?](https://substack.com/sign-in?redirect=%2Faccount%3Futm_source%3Dsubstack%26utm_medium%3Demail&for_pub=mindfulmodeler&email=&with_password=&change_user=false&justTrying=)), el autor del maravilloso libro gratuito sobre XAI [_"Interpretable Machine Learning"_](https://christophm.github.io/interpretable-ml-book/), que recomendamos en el Jupyter Notebook `XAI.ipynb`, que ahora se ha pasado a la UQ y ha publicado un libro sobre Conformal Prediction <font size="3">??????</font>  (enlace m?s abajo).
+ El genial post sobre c?mo usan Conformal Prediction para conformalizar la clasificaci?n de sus clientes en el [banco BBVA](https://www.bbvaaifactory.com/es/conformal-prediction-an-introduction-to-measuring-uncertainty/)
+ La maravillosa charla _"Quantifying uncertainty in Machine Learning predictions"_ de [Maria Navarro](https://youtu.be/r6bhm_A-YcQ), de las primeras en implementar CP en la industria.
+ La charla sobre Conformal Predictors en el [curso sobre incertidumbre de la Universidad de Pennsylvania](https://uncertaintyclass.com/), este es el [v?deo](https://youtu.be/M3tkM4dcIPA).
+ El blog de los creadores de los 'Conformal Predictors'  [Prof. Vladimir Vovk y Prof. Alex Gammerman:](https://cml.rhul.ac.uk/cp.html).
+ Este [kernel de Kaggle](https://www.kaggle.com/code/carlmcbrideellis/regression-prediction-intervals-with-mapie) que explica c?mo usar los Conformal Predictors en una tarea de regresi?n.
+ Teor?a y ejemplos en el repositorio [A. Angelopoulos](https://github.com/aangelopoulos/conformal-prediction), uno de los dos autores de [_"A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification_"](https://arxiv.org/abs/2107.07511).
    
<font color=#6B8E23 size="3"> Libros

+ El libro de Christoph Molnar [_"Introduction To Conformal Prediction With Python. A Short Guide For Quantifying Uncertainty Of Machine Learning Models"_](https://christophmolnar.com/books/conformal-prediction/).    
+ El nuevo libro de Valeriy Manokhin [_"Practical Guide to Applied Conformal Prediction in Python: Learn and apply the best uncertainty frameworks to your industry applications "_](https://www.packtpub.com/product/practical-guide-to-applied-conformal-prediction-in-python/9781805122760).
+ El libro de los creadores de la Predicci?n Conforme Vladimir Vovk, Alexander Gammerman, Glenn Shafer ["_Algorithmic Learning in a Random World_"](https://link.springer.com/book/10.1007/b106715)

<font color=#6B8E23 size="3">Paquetes de Python especializados en Conformal Prediction listos para un `pip install`</font>  
+ [MAPIE](https://github.com/scikit-learn-contrib/MAPIE), desarrollada por especialistas en Visi?n Artificial, incluye algoritmos de CP para clasificaci?n, regresi?n, y time series,
+ [CREPES](https://github.com/henrikbostrom/crepes), incluye algoritmos de CP para clasificaci?n, regresi?n, y una implementaci?n de CP llamada "Conformal Predictive Distributions" que produce la cumulative distribution function de cada predici?n.
+ [R2CCP](https://github.com/EtashGuha/R2CCP), especializada en regresi?n, calcula la incertidumbre incluso si los datos no est?n siempre igual de esparcidos (heterocedasticidad) o su distribuci?n es bimodal,
+ [PUNCC](https://github.com/deel-ai/puncc), icluye CP para regresi?n, clasificaci?n y detecci?n de anomal?as,
+ [TorchCP](https://github.com/ml-stat-Sustech/TorchCP) especialmente para Deep Learning, 
+ [Conformal Tights](https://pypi.org/project/conformal-tights/) especializada en problemas de regresi?n, y
+ [nonconformist](https://github.com/donlnz/nonconformist/blob/master/README.ipynb), el primero que hubo, incluye una implementaci?n de la "Transductive Conformal Prediction".
    


<a name='uq-ivap'/>

### <font color=#ac6240>4.2  Intervalos de predicci?n de fallo card?aco con Inductive Venn-ABERS Prediction</font>

Apliquemos la 'Conformal Prediction' a las predicciones de nuestro Bosque Aleatorio. 

Las clasificaciones binarias son especiales porque los conjuntos de predicci?n generados para las predicciones inciertas no ser?n muy informativos, simplemente contendr?n ambas clases. Es decir, para una predicci?n incierta obtendremos ['0','1'] y esta no aporta mucho. Por ello para el caso binario en lugar de aplicar Inductive (tambi?n llamada SPlit) Conformal Prediction para crear conjuntos de predicci?n para las clases, vamos a aplicar la conformalizaci?n que crea intervalos de predicci?n para las probabilidades, de manera que una predicci?n incierta obtendr? un intervalo de probabilidad ancho [0.4, 0.9] y una confiable obtendr? un intervalo de probabilidad estrecho, por ejemplo, para une paciente con poco riesgo ser?a [0.1,0.2]. Aplicamos entonces la denominada 'Inductive Venn-ABERS Prediction (IVAP)' descrita en el art?culo [_"Large-scale probabilistic prediction with and without validity guarantees"_](http://alrw.net/articles/13.pdf) y aplicada en este notebook de Kaggle del grandmaster Carl McBride Ellis [Classifier calibration Using Venn-ABERS](https://www.kaggle.com/code/carlmcbrideellis/classifier-calibration-using-venn-abers). Es un tipo un poco especial de CP porque no requiere la definici?n de una medida de no conformidad per se, ni de un nivel de confianza para la garant?a de cobertura. Consiste en calibrar los resultados con una [Regresi?n Isot?nica](https://docs.google.com/presentation/d/1Q6oxcgmNv0GsmNFA5npAzQGDuy_XolBKStN4KUhP4Gk/edit#slide=id.g2c46ccaba95_0_177) para cada posible clase. Es decir, a cada nueve paciente le otorga primero la clase 0 y calcula si es un resultado inusual o no y luego le otorga la clase 1 y calcula si este otro es tambi?n un resultado inusual o no. As? obtenemos los l?mites superior (p1) e inferior (p0) del intervalo de predicci?n. Para reunir la informaci?n en una sola probabilidad, se aplica la f?rmula que minimiza la log-loss:  `p_calibrada = p1/(1 - p0 + p1)`.

Este m?todo, al ser "Inductive" requiere de un conjunto de datos de calibraci?n que ya separamos al comienzo del este Jupyter Notebook:

In [ ]:
# preparation CALIBRATION data

# reset index after the splitting shuffling
df_cal = df_cal.reset_index(drop=True)
# define target of the test set
y_cal = df_cal.HeartDisease.values
   
# 1. encode the categorical features and remove redundant columns-------------    

cal_dict = df_cal[categorical].to_dict(orient='records')
X_cal_cat = dv.transform(cal_dict) 

# convert the data array to a dataframe
df_X_cal_cat = pd.DataFrame(X_cal_cat,columns= dv.get_feature_names_out())

# remove redundant columns
df_X_cal_cat.drop(['ExerciseAngina=N', 'FastingBS=H', 'Sex=F', 'ST_Slope=Up'], 
                    axis=1, inplace=True)

# convert the dataframe to a np.array again
X_cal_cat = df_X_cal_cat.to_numpy()

# 2. scale the numerical features --------------------------------------------

X_cal_num = df_cal[numerical].values
#X_cal_num = scaler.transform(X_cal_num) 

# 3. join the matrices -------------------------------------------------------

X_cal = np.column_stack([X_cal_num, X_cal_cat]) 

Usamos la implementaci?n del repositorio de [Paolo Toccaceli](https://github.com/ptocca/VennABERS/), que para cada paciente nos calcular? los l?mites superior e inferior de los intervalos de predicci?n y la probabilidad conformalizada. Simplemente copiamos y pegamos aqu? el c?digo:

In [ ]:
# Some elementary functions to speak the same language as the paper
# (at some point we'll just replace the occurrence of the calls with the function body itself)
def push(x,stack):
    stack.append(x)
    
def pop(stack):
    return stack.pop()

def top(stack):
    return stack[-1]

def nextToTop(stack):
    return stack[-2]

# perhaps inefficient but clear implementation
def nonleftTurn(a,b,c):   
    d1 = b-a
    d2 = c-b
    return np.cross(d1,d2)<=0

def nonrightTurn(a,b,c):   
    d1 = b-a
    d2 = c-b
    return np.cross(d1,d2)>=0

def slope(a,b):
    ax,ay = a
    bx,by = b
    return (by-ay)/(bx-ax)

def notBelow(t,p1,p2):
    p1x,p1y = p1
    p2x,p2y = p2
    tx,ty = t
    m = (p2y-p1y)/(p2x-p1x)
    b = (p2x*p1y - p1x*p2y)/(p2x-p1x)
    return (ty >= tx*m+b)

kPrime = None

# Because we cannot have negative indices in Python (they have another meaning), I use a dictionary

def algorithm1(P):
    global kPrime
    
    S = []
    P[-1] = np.array((-1,-1))
    push(P[-1],S)
    push(P[0],S)
    for i in range(1,kPrime+1):
        while len(S)>1 and nonleftTurn(nextToTop(S),top(S),P[i]):
            pop(S)
        push(P[i],S)
    return S

def algorithm2(P,S):
    global kPrime
    
    Sprime = S[::-1]     # reverse the stack

    F1 = np.zeros((kPrime+1,))
    for i in range(1,kPrime+1):
        F1[i] = slope(top(Sprime),nextToTop(Sprime))
        P[i-1] = P[i-2]+P[i]-P[i-1]
        if notBelow(P[i-1],top(Sprime),nextToTop(Sprime)):
            continue
        pop(Sprime)
        while len(Sprime)>1 and nonleftTurn(P[i-1],top(Sprime),nextToTop(Sprime)):
            pop(Sprime)
        push(P[i-1],Sprime)
    return F1

def algorithm3(P):
    global kPrime

    S = []
    push(P[kPrime+1],S)
    push(P[kPrime],S)
    for i in range(kPrime-1,0-1,-1):  # k'-1,k'-2,...,0
        while len(S)>1 and nonrightTurn(nextToTop(S),top(S),P[i]):
            pop(S)
        push(P[i],S)
    return S

def algorithm4(P,S):
    global kPrime
    
    Sprime = S[::-1]     # reverse the stack
    
    F0 = np.zeros((kPrime+1,))
    for i in range(kPrime,1-1,-1):   # k',k'-1,...,1
        F0[i] = slope(top(Sprime),nextToTop(Sprime))
        P[i] = P[i-1]+P[i+1]-P[i]
        if notBelow(P[i],top(Sprime),nextToTop(Sprime)):
            continue
        pop(Sprime)
        while len(Sprime)>1 and nonrightTurn(P[i],top(Sprime),nextToTop(Sprime)):
            pop(Sprime)
        push(P[i],Sprime)
    return F0

def prepareData(calibrPoints):
    global kPrime
    
    ptsSorted = sorted(calibrPoints)
    
    xs = np.fromiter((p[0] for p in ptsSorted),float)
    ys = np.fromiter((p[1] for p in ptsSorted),float)
    ptsUnique,ptsIndex,ptsInverse,ptsCounts = np.unique(xs, 
                                                        return_index=True,
                                                        return_counts=True,
                                                        return_inverse=True)
    a = np.zeros(ptsUnique.shape)
    np.add.at(a,ptsInverse,ys)
    # now a contains the sums of ys for each unique value of the objects
    
    w = ptsCounts
    yPrime = a/w
    yCsd = np.cumsum(w*yPrime)   # Might as well do just np.cumsum(a)
    xPrime = np.cumsum(w)
    kPrime = len(xPrime)
    
    return yPrime,yCsd,xPrime,ptsUnique

def computeF(xPrime,yCsd):
    global kPrime
    P = {0:np.array((0,0))}
    P.update({i+1:np.array((k,v)) for i,(k,v) in enumerate(zip(xPrime,yCsd))})
    
    S = algorithm1(P)
    F1 = algorithm2(P,S)
    
    P = {0:np.array((0,0))}
    P.update({i+1:np.array((k,v)) for i,(k,v) in enumerate(zip(xPrime,yCsd))})    
    P[kPrime+1] = P[kPrime] + np.array((1.0,0.0))    # The paper says (1,1)
    
    S = algorithm3(P)
    F0 = algorithm4(P,S)
    
    return F0,F1

def getFVal(F0,F1,ptsUnique,testObjects):
    pos0 = np.searchsorted(ptsUnique,testObjects,side='left')
    pos1 = np.searchsorted(ptsUnique[:-1],testObjects,side='right')+1
    return F0[pos0],F1[pos1]

def ScoresToMultiProbs(calibrPoints,testObjects):
    # sort the points, transform into unique objects, with weights and updated values
    yPrime,yCsd,xPrime,ptsUnique = prepareData(calibrPoints)
    
    # compute the F0 and F1 functions from the CSD
    F0,F1 = computeF(xPrime,yCsd)
    
    # compute the values for the given test objects
    p0,p1 = getFVal(F0,F1,ptsUnique,testObjects)
                    
    return p0,p1

Aplicando la funci?n `ScoresToMultiProbs()` a  la lista de pares '(raw_score, true value)' de las predicciones del Bosque Aleatorio que obtuvimos sobre el conjunto de entrenamiento, y la 'raw score' de la instancia dada (une nueve paciente, por ejemplo del conjunto de prueba),  obtenemos el l?mite inferior 'p0' y superior 'p1' de nuestro intervalos de predicci?n para la probabilidad bien calibrada para la clase '1', en nuestro caso la probabilidad de pertenecer al grupo en riesgo de sufrir fallo card?aco. Como dec?amos m?s arriba, la probabilidad calibrada de 'HeartDisease=1' para ese paciente se calcula con la f?rmula `p_calibrada = p1/(1 - p0 + p1)` que se obtiene a partir de los l?mites de nuestro intervalo.

Apliquemos la conformalizaci?n a todo el conjunto de prueba:

In [ ]:
# calibrPts: a list of pairs (score,label) corresponding to the scores and labels of the calibration examples.
y_pred_cal = RF.predict_proba(X_cal)[:,1] 
calibrPts = zip(list(y_pred_cal),list(y_cal))

# testScores: a list of floats corresponding to the scores for the test objects.
testScores = RF.predict_proba(X_test)[:,1]  

# interval bounds
p0,p1 = ScoresToMultiProbs(calibrPts,testScores)

# united probability that minimizes the log loss https://towardsdatascience.com/intuition-behind-log-loss-score-4e0c9979680a 
p = p1/(1-p0+p1) 

predictions = pd.Series(y_test).to_frame()
predictions.columns = ['y_true']
predictions["raw scores"] = (testScores)# >= 0.4)
predictions["p0"] = p0 # lower bound
predictions["p1"] = p1 # upper bound
predictions["p"]  = p # estimated probability
predictions["width"] = predictions["p1"] - predictions["p0"] # interval for UQ
predictions = predictions.sort_values(by=['p'])
predictions = predictions.reset_index()

plt.style.use('fivethirtyeight')

fig, ax = plt.subplots(figsize=(12, 6))
s =30
plt.scatter(x=predictions.index,y=predictions["raw scores"],label="non-calibrated", marker = 'o', s = s + 250 , color = 'c')
plt.scatter(x=predictions.index,y=predictions["p1"],label="p1 upper probability",s=s +70)
plt.scatter(x=predictions.index,y=predictions["p0"],label="p0 lower probability",s=s +70)
plt.scatter(x=predictions.index,y=predictions["p"],label="conformal probability",s=s +30, marker = 'o')
plt.scatter(x=predictions.index,y=predictions["y_true"],label="true label",s= s, marker = 'o', color = "k")

plt.plot(predictions.index,predictions["width"],label="p1-p0 interval width",lw=3.5,color="pink")

# decision thershold
plt.hlines(y=0.4, xmin=0, xmax=30, lw=0.5, color='red')
plt.text(1, 0.45, 'y=0.4', ha='right', va='center', color='red')
# 90% confidence in intervals
plt.hlines(y=0.05, xmin=0, xmax=30, lw=0.5, color='red')
plt.text(1, 0.15, 'y=0.05', ha='right', va='center', color='red')

#plt.legend(loc="upper left",fontsize=16, scatterpoints=1, markerscale=1)
plt.legend(fontsize=15, scatterpoints=1, markerscale=1)

plt.ylabel("probability",fontsize=30)
plt.xlabel("test set sorted by the conformal probability", fontsize=30)
plt.xticks([])
plt.rc('ytick', labelsize= 15)
plt.rc('xtick', labelsize= 15)
#plt.savefig('All_test_probabilities.pdf')
plt.show()

Podemos decidir no automatizar aquellas cuyo intervalo sea muy grande, m?s grande que el nivel de confianza $\epsilon$, por ejemplo 0.05, para una confianza del 95%.

Vemos que la Prediccion Conforme en este caso no consigue reclasificar correctamente los 2 Falsos Negativos pero nos ofrece intervalos de predicci?n mayores que el riesgo acceptado. Con la anchura mayor de 0.05, la conformalizaci?n nos indica que no debemos confiar en estas predicciones.

Por desgracia ambos Falsos Positivos tienen un intervalo de predicci?n es muy estrechito, la confomalizaci?n indica err?neamente que es un resultado confiable. La Predicci?n Conforme sobre el Bosque Aleatorio no ha sido lo suficientemente potente para identificar lo inusual de estos resultado y los clasifica como confiables a pesar de ser err?neos. s?lo un nivel de confianza del 99% (i.e., $\epsilon$ = 0.01) habr?a detectado estos casos. La clasificaci?n de estes pacientes es particularmenete complicada. C?mo siempre, aumentar el tama?o de los conjuntos de entrenamiento y calibraci?n tambi?n puede que mejorara los resultados, no tanto por la parte de la conformalizaci?n sino porque aumentar?a el rendimiento del Bosque Aleatorio.

#### Evaluaci?n de la conformalizaci?n con IVAP

In [ ]:
# performance metrics with decision threshold t = 0.4
acc = accuracy_score(y_test, p >= 0.4)
f1 = f1_score(y_test, p >= 0.4)

# calibration metrics after conformalizing with IVAP
l_loss = log_loss(y_test, p)
brier_loss = brier_score_loss(y_test, p)

metrics_list.append(["Random Forest + IVAP", acc.round(3), f1.round(3), brier_loss.round(3), l_loss.round(3)])


metrics_df = pd.DataFrame(metrics_list, columns = [
    'Classifier', 'W-accuracy','F1-score', 'Brier Score', 'Log Loss'])
metrics_df.set_index('Classifier', inplace=True)
metrics_df.columns = pd.MultiIndex.from_tuples([ 
                              ('   Performance metrics:', 'Weighted Acc'),('   Performance metrics:', 'F1-score'), 
                              ('Calibration metrics:', 'Brier Score'),('Calibration metrics:', 'Log Loss')])

metrics_df.round(2)

La conformalizaci?n con IVAP nos da intervalos de predicci?n pero esta vez no ha mejorado la calibraci?n ni el rendimiento, lo cual no es lo que quer?amos mostrar y que habr?a que investigar por qu? hemos obtenido este resultado tan inusual, que, c?mo dec?amos, puede deberse al peque?o volumen o las car?cteristica del conjunto de datos que hacen que el modelo clasificador sea menos efectivo. Porque hemos de tener en cuenta que la calidad de la UQ depende de la calidad del model clasificador cuya incertidumbre cuantifica.

Para ver ejemplos donde la IVAP muestra todo su pontencial descubriendo Falsos Negativos en medicina y calibrando mejor que otros m?todos de calibraci?n populares ([Platt Scaling](https://scikit-learn.org/stable/modules/calibration.html#sigmoid) e [Isotonic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.isotonic.IsotonicRegression.html), que no ofrecen garant?a de cobertura) ver por ejemplo:

+ Este [Kaggle Notebook de Carl McBride Ellis](https://www.kaggle.com/code/carlmcbrideellis/classifier-calibration-using-venn-abers/notebook).
+ Este [Jupyter Notebook de Xema Belmonte Mar?n](https://github.com/xemauoc/TFG/blob/main/TFG_04_Incertidumbre.ipynb) para su [Trabajo de Fin de Grado](https://openaccess.uoc.edu/items/7b4f93a5-a0b7-4c11-b8ec-89530b999ecc).
+ Este [Jupyter Notebook de Pablo Pim?s Verge](https://github.com/ppimas-uoc/TFG---Inteligencia-Artificial/blob/master/Fase%203/4_Incertidumbre.ipynb) para su [Trabajo de Fin de Grado](https://openaccess.uoc.edu/items/ad596628-3be9-4f89-9ecf-4ade9ba7d35e).
+ Este [Jupyter Notebook de Pablo V?zquez Rodr?guez](https://github.com/pvazquezr/TFG_AI/blob/master/Project/02_3_Modelado_Calibraci%C3%B3n_de_modelos.ipynb) para su [Trabajo de Fin de Grado](https://openaccess.uoc.edu/items/1b3a1533-a17c-4c44-a016-42598cb5274c).

Las predicciones con m?s incerticumbre tienen los intervalos de predicci?n son m?s grandes. Podemos decidir no automatizar aquellas cuyo intervalo sea muy grande, m?s grandes que el nivel de confianza' $\epsilon$, por ejemplo 0.05, para una confianza del 95%. En un ejemplo real implementaremos una se?al de fallback y el programa nos dir? 'no puedo predecir, no tengo ni idea' y puede ser preferible una inversi?n en tiempo humano para la clasificaci?n manual de esa instancia que asumir las clasificaciones incorrectas. Por ejemplo, en nuestro caso, une experte en medicina evaluar?a a eses pacientes seleccionados gracias a la cuantifici?n de incertidumbre y les dirigir?a a hacerse nuevas pruebas m?dicas, por ejemplo.

<a name='uq-others'/>

### <font color=#ac6240>4.3 Otros m?todos de UQ para aprendizaje supervisado</font>

Hay miembros de la familia de la Predicci?n Conforme para todo tipo de tareas y modelos de aprendizaje autom?tico. Sin embargo, al contrario que la mayor?a de los m?todos de explicabilidad que mostramos en el Jupyter Notebook anterior `XAI.ipynb`, muchos m?todos de UQ en general y de CP en particular son espec?ficos, por ejemplo, se aplican s?lo a tareas de regresi?n. Mencionaremos algunos muy populares y cabe mencionar que:

+ todos los m?todos de UQ que no usen CP se deben calibrar con 'Conformal Predictions' o estar?mos en las mismas que cuando s?lo nos preocupa el renedimiento y usamos s?lo los 'raw scores' de `.predict_proba()`.

+ s?lo la Conformal Prediction nos da garant?a de cobertura de que el resultado contiene el valor verdadero. Por ejemplo, la primera columna de siguiente tabla del art?culo [Valid prediction intervals for regression problems](https://link.springer.com/article/10.1007/s10462-022-10178-5) (en Arxiv est? la [?ltima versi?n](https://arxiv.org/abs/2107.00363)) d?nde se revisan varios m?todos de UQ para regresi?n.

![Waegeman_benchmark](./images/Waegeman_UQ_benchmark.png "Waegeman_benchmark")

[Fuente de la imagen: Valid prediction intervals for regression problems](https://link.springer.com/article/10.1007/s10462-022-10178-5), en Arxiv est? la [?ltima versi?n](https://arxiv.org/abs/2107.00363). 

<font color=#ac6240>Paquetes que implementan CP y otros m?todos de UQ para cualquier tarea </font>
+ [PiML](https://selfexplainml.github.io/PiML-Toolbox/_build/html/guides/testing/reliability.html)
+ [AWS Fortuna](https://github.com/awslabs/fortuna?tab=readme-ov-file) (para CP con time series soluciona el asunto de la intercambiabilidad implementando [EnbPI](https://proceedings.mlr.press/v139/xu21h/xu21h.pdf), como hace MAPIE)
+ [Lightning UQ Box](https://github.com/lightning-uq-box/lightning-uq-box) 

<font color=#ac6240>Tareas de Regresi?n</font>

El art?culo y el Jupyter Notebook del [tutorial de Martim Sousa](https://github.com/Quilograma/ConformalPredictionTutorial/tree/main) (tambi?n incluye la explicaci?n para la Quantile Regression, ver siguiente punto aqu? abajo).


<font color=#ac6240>Quantile Regression</font>

Pertenece al grupo "Direct Interval Estimation" de la tabla anterior.

+ Una breve y genial explicaci?n de la [Quantile Regression escrita por Christoph Molnar](https://mindfulmodeler.substack.com/p/how-i-made-peace-with-quantile-regression) y sobre como [conformalizarla aqu?](https://mindfulmodeler.substack.com/p/week-3-conformal-prediction-for-regression).
+ M?s sobre conformalizar la Quantile Regression en [_"A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification"_](https://arxiv.org/pdf/2107.07511.pdf) secci?n "2. Examples of Conformal Procedures" y [_"Another (Conformal) Way to Predict Probability Distributions: Conformal multi-quantile regression with Catboost"_](https://towardsdatascience.com/another-conformal-way-to-predict-probability-distributions-fcc63e78680d).

<font color=#ac6240>Ensembles</font>

Suelen ser computacionalmente costosos porque implican entrenar muchas versiones de los modelos de aprendizaje autom?tico y no ofrecen garant?a de cobertura. Un ejemplo de aplicaci?n:  [_"Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles"_](https://papers.nips.cc/paper/2017/hash/9ef2ed4b7fd2c810847ffa5fa85bce38-Abstract.html).

<font color=#ac6240>Bayesianos</font>

Suelen ser computacionalmente costosos porque implican predecir la distribuci?n completa, siempre dependen de un prior que puede ser dif?cil de estimar y por lo tanto no ser fiable, y no garantizan la cobertura. Para que los resultados de los m?todos bayesianos cumplan "validity" pueden conformalizarse, ver por ejemplo:

+ [Valid prediction intervals for regression problems](https://link.springer.com/article/10.1007/s10462-022-10178-5) (en Arxiv est? la [?ltima versi?n](https://arxiv.org/abs/2107.00363)), 
+ [Bayesian Optimization with Formal Safety Guarantees via Online Conformal Prediction](https://arxiv.org/abs/2306.17815), and 
+ [Online Calibrated and Conformal Prediction Improves Bayesian Optimization](https://proceedings.mlr.press/v238/deshpande24a/deshpande24a.pdf).

Sobre m?todos bayesianos en general:

+ Este art?culo en este blog: [El Paradigma Bayes para escenarios de incertidumbre](https://keepler.io/es/2020/03/el-paradigma-bayes-para-escenarios-de-incertidumbre/).

+ La librer?a de Python [PyMC3](https://pypi.org/project/pymc3/) (que ahora se llama [PyMC](https://www.pymc.io/blog/v4_announcement.html)) y el repo [Probabilistic Python: An Introduction to Bayesian Modeling with PyMC](https://github.com/fonnesbeck/probabilistic_python).

+ M?todos Bayesianos para clasificaci?n y regresi?n como BART [Bayesian Additive Regression Trees](https://jmloyola.github.io/posts/2019/06/introduction-to-bart)

+ Para 'Gaussian Processes', ver por ejemplo: [_"Easy introduction to gaussian process regression (uncertainty models)"_](https://youtu.be/iDzaoEwd0N0) y [_"Coding gaussian process regressors FROM SCRATCH in python"_](https://youtu.be/JXdrq7--XV0) con este [repo](https://gitlab.com/youtube-optimization-geeks/uncertainty-quantification/-/blob/main/coing_gpr_from_scratch_notebook.ipynb).

+ Para Deep Learning, la charla _"Are you sure about that?! Uncertainty Quantification in AI"_ de [Florian Wilhelm](https://youtu.be/LCDIqL-8bHs) con aplicaciones en Deep Learning para tareas de regresi?n.

<font color=#ac6240>Para la predicci?n de Time Series, donde los datos no son intercambiables</font>

Exiten varios algoritmos de CP para calcular intervalos de predicci?n espec?ficos para tareas de forecasting como [EnbPI](https://proceedings.mlr.press/v139/xu21h/xu21h.pdf), [ACI](https://arxiv.org/abs/2106.00170), o [CFTS](https://proceedings.neurips.cc/paper_files/paper/2021/file/312f1ba2a72318edaaa995a67835fad5-Paper.pdf) y varios paquetes que las incluyen:

+ La explicaci?n de c?mo usar la libreria `MAPIE` con los Conformal Predictors para UQ en time series: [_"Time Series Forecasting with Conformal Prediction Intervals: Scikit-Learn is All you Need"_](https://towardsdatascience.com/time-series-forecasting-with-conformal-prediction-intervals-scikit-learn-is-all-you-need-4b68143a027a). Soluciona el asunto de la intercambiabilidad implementando EnbPI, como hace AWS Fortuna y tambi?n ACI. 

+ [MLforecast](https://nixtlaverse.nixtla.io/mlforecast/docs/how-to-guides/prediction_intervals.html) y el modelo fundacional [TimeGPT](https://www.nixtla.io/docs/forecasting/probabilistic/prediction_intervals) de Nixtla incluyen CP. Solucionan el asunto de la incercambiabilidad implementando CFTS.

+ [Sktime](https://www.sktime.net/en/stable/examples/01b_forecasting_proba.html) Tambi?n solucionan el asunto de la incercambiabilidad implementando CFTS.

Tambi?n pueden aplicarse simplemente la Inductive (tambi?n llamada Split) Conformal Prediction directamente sobre una Quantile Regressi?n para obtener Conformalized [Quantile Regression o CQR](https://arxiv.org/abs/1905.03222), ver por ejemplo:

+ [NeuralProphet](https://pypi.org/project/neuralprophet/) tiene implementada la funci?n `.conformal_predict()` para conformalizar la Quantile Regression en time series, [este Jupyter Notebook](https://github.com/ourownstory/neural_prophet/blob/main/tutorials/feature-use/uncertainty_conformal_prediction.ipynb) contiene un ejemplo prediciendo la demanda de energia de un hospital, y
+ Este tutorial de [Skforecast](https://skforecast.org/0.15.1/user_guides/probabilistic-forecasting-conformal-prediction.html)

En esta presentaci?n [_"Conformal Prediction for Time Series Forecasting"_](https://docs.google.com/presentation/d/1oVezKS5pimcQq21-slBHRq-lJBUNftyZENX3JGoNzXo/edit#slide=id.g2e4d971ff01_0_14) inclu?mos una lista con todas las fuentes actualizadas y una breve explicaci?n de algunos algoritmos de CP para time series forecasting.

En este [?ngela Manzanares Alarcia para su Trabajo de Fin de Grado](https://openaccess.uoc.edu/items/5e7bfba9-f4bd-4086-a868-072ce0a6ff56) cre? este [repositorio aplicando las CQR, EnbPI y ACI de MAPIE](https://github.com/amanzalar/TFG-Demanda/tree/main/notebooks).

<a name='miscelanea'/>

### <font color=#ac6240>4.3 Miscel?nea</font>

<font color=#ac6240>GAMLSS</font>

Los Generalized Additive Models for Location, Scale and Shape son modelos de regresi?n que no predicen los valores de la variable de destino o 'target' sino que predicen los parametros que definen su distribuci?n, es decir, la localizaci?n (por ejemplo, la media), la escala (por ejemplo, la varianza), y la forma (por ejemplo, la kurtosis y la 'skewness'), m?s info en [este art?culo](https://www.cienciadedatos.net/documentos/63_gamlss.html).    

La librer?a [Lightning UQ Box](https://github.com/lightning-uq-box/lightning-uq-box) implementa la "Mean-Variance Estimation" donde el resultado de la red neuronal es la media y la varianza de la distribuci?n de la predicci?n pero asumiendo entonces que los resultados est?n distribuidos seg?n una distribuci?n normal (recordamos que los Conformal Predictors no son param?tricos, son distribution-free).

<font color=#ac6240>Trustworthy AI: UQ + XAI</font>

Podemos aplicar UQ no s?lo a los resultados de la predicci?n sino a los resultados de la XAI que obtuvimos en el anterior Jupyter Notebook sobre explicabilidad en este taller y dotar de intervalos de predicci?n a los Shapley Values, como en este art?culo: [But Are You Sure? An Uncertainty-Aware Perspective on Explainable AI](https://proceedings.mlr.press/v206/marx23a/marx23a.pdf) o como en este art?culo sobre  [conformalizar los valores de Shapley](https://raw.githubusercontent.com/mlresearch/v230/main/assets/alkhatib24a/alkhatib24a.pdf).

<font color=#ac6240>Causal Inference</font>

+ Blogs como [Causal Inference for The Brave and True](https://matheusfacure.github.io/python-causality-handbook/landing-page.html) 
+ La charla _"Introducci?n a la causalidad en Python "_ de [Albert Pujol Torras y Miguel F. Alarc?n](https://charlas.2022.es.pycon.org/pycones2022/talk/FTGTLS/) en la PyConES 2022.

Los m?todos de causalidad tambi?n han de conformalizarse, por ejemplo:

+ [Conformal Counterfactual Inference under Hidden Confounding](https://arxiv.org/abs/2405.12387)

+ [Conformal convolution and Monte Carlo meta-learners for predictive inference of individual treatment effects](https://arxiv.org/pdf/2402.04906)

+ [Conformal Prediction for Causal Effects of Continuous Treatments](https://arxiv.org/abs/2407.03094)

+ [On the Role of Surrogates in Conformal Inference of Individual Causal Effects](https://arxiv.org/abs/2412.12365)

<font color=#ac6240>Modelos h?bridos</font>

Son aquellos que incluyen 'domain expertise' para comprobar que se cumplen las restricciones que tenga el sistema, (por ejemplo las leyes de conservaci?n de la energ?a y la masa, limitaciones de presupuesto,...) o para 'feature engineering'; un ejemplo son las Physic Guided o [Physic Informed Neural Networks](https://en.wikipedia.org/wiki/Physics-informed_neural_networks) como en el art?culo [_"Physics-guided Neural Networks (PGNN): An Application in Lake Temperature Modeling"_](https://arxiv.org/pdf/1710.11431.pdf) un ejemplo de una red neuronal artificial aplicada en hidrolog?a que arrojaba soluciones que permit?an que agua l?quida m?s densa flotara sobre agua l?quida menos densa. 

**Si es que si no se lo dices, todo esto al algoritmo le da igual**.


<font size="6">???????</font>

La librer?a [SciANN](https://www.sciann.com/#new-to-sciann) permite modificar la loss function de las redes neuronales para penalizar aquellas soluciones que violen las limitaciones o restricciones del sistema.

Un ejemplo de la conformalizaci?n de una PINN: [_"Conformalized Physics-Informed Neural Networks"_](https://arxiv.org/abs/2405.08111).

<a href=#toc>Subir a Tabla de Contenidos</a><br>

<a name='uq-end'/>

### <font color=#ac6240>5. Conclusi?n sobre la cuantificaci?n de incertidumbre</font>

Aplicando Conformal Prediction a nuestros resultados hemos conseguido identificar la aquellos pacientes sobre cuya clasificacion el bosque aleatorio no estaba muy seguro. Podemos entonces automatizar s?lo las predicciones seguras y reducir as? nuestra cantidad de Falsos Negativos y Positivos. 

Vemos que son m?todos f?ciles de implementar y r?pidos, y nos proporcionan probabilidades rigurosas.


<font size="5">??????????????????? ??? ????</font>

<a href=#toc>Subir a Tabla de Contenidos</a><br>

Nos vemos en el pr?ximo taller sobre Predicci?n Conforme para clasificiaci?n multiclase que encontrar?s en el Jupyter Notebook `UQ_multiclass.ipynb` en este mismo repositorio. 

<font size="5">???? </font> Buen trabajo y hasta pronto!

